In [ ]:
import time
import matplotlib

# Disable interactive rendering for performance
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# ----------------------------------------
# CONFIG
# ----------------------------------------

plt.style.use("ggplot")

BASE_DIR = Path("../")
EXPORTS_DIR = BASE_DIR / "exports"
CHARTS_DIR = BASE_DIR / "charts"

CHARTS_DIR.mkdir(exist_ok=True)

start_time = time.time()

print("Starting EDA pipeline...")
print("-" * 50)

# ----------------------------------------
# LOAD DATASETS
# ----------------------------------------

print("Loading timeline dataset...")
timeline_df = pd.read_csv(
    EXPORTS_DIR / "timeline_clean.csv"
)
print("Timeline loaded")

print("Loading ratings dataset...")
ratings_df = pd.read_csv(
    EXPORTS_DIR / "ratings_clean.csv"
)
print("Ratings loaded")

print("Loading reviews dataset...")
reviews_df = pd.read_csv(
    EXPORTS_DIR / "reviews_clean.csv"
)
print("Reviews loaded")

print("Loading ranks dataset...")
ranks_df = pd.read_csv(
    EXPORTS_DIR / "ranks_clean.csv"
)
print("Ranks loaded")

# ----------------------------------------
# CONVERT DATES
# ----------------------------------------

print("\nConverting date columns...")

timeline_df["date"] = pd.to_datetime(
    timeline_df["date"]
)

ratings_df["date"] = pd.to_datetime(
    ratings_df["date"]
)

reviews_df["date"] = pd.to_datetime(
    reviews_df["date"]
)

ranks_df["date"] = pd.to_datetime(
    ranks_df["date"]
)

print("Date conversion completed")

# ----------------------------------------
# TIMELINE EVENTS
# ----------------------------------------

print("\nAnalyzing timeline events...")

event_counts = (
    timeline_df["event_type_name"]
    .value_counts()
)

print(event_counts)

plt.figure(figsize=(10, 5))

event_counts.plot(kind="bar")

plt.title("Timeline Event Distribution")
plt.xlabel("Event Type")
plt.ylabel("Count")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "timeline_event_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Timeline chart exported")

# ----------------------------------------
# GROSSING TREND
# ----------------------------------------

print("\nAnalyzing grossing rank trend...")

grossing_trend = (
    ranks_df
    .groupby("date")[
        "store_product_rank_grossing"
    ]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 6))

plt.plot(
    grossing_trend["date"],
    grossing_trend[
        "store_product_rank_grossing"
    ]
)

plt.gca().invert_yaxis()

plt.title("Grossing Rank Trend Over Time")
plt.xlabel("Date")
plt.ylabel("Grossing Rank")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "grossing_rank_trend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Grossing rank chart exported")

# ----------------------------------------
# RATING EVOLUTION
# ----------------------------------------

print("\nAnalyzing ratings evolution...")

rating_trend = (
    ratings_df
    .groupby("date")[
        "average_star_cumulative"
    ]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 6))

plt.plot(
    rating_trend["date"],
    rating_trend[
        "average_star_cumulative"
    ]
)

plt.title("Average Rating Evolution")
plt.xlabel("Date")
plt.ylabel("Average Rating")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "average_rating_evolution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Rating evolution chart exported")

# ----------------------------------------
# REVIEW SENTIMENT
# ----------------------------------------

print("\nAnalyzing review sentiment...")

sentiment_counts = (
    reviews_df["review_sentiment"]
    .value_counts()
)

print(sentiment_counts)

plt.figure(figsize=(8, 5))

sentiment_counts.plot(kind="bar")

plt.title("Review Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

plt.xticks(rotation=0)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "review_sentiment_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Sentiment chart exported")

# ----------------------------------------
# TOP REVIEW TOPICS
# ----------------------------------------

print("\nAnalyzing top review topics...")

top_topics = (
    reviews_df["topics"]
    .value_counts()
    .head(10)
)

print(top_topics)

plt.figure(figsize=(10, 6))

top_topics.plot(kind="barh")

plt.title("Top Review Topics")
plt.xlabel("Count")
plt.ylabel("Topic")

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "top_review_topics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Top topics chart exported")

# ----------------------------------------
# EVENTS VS RANKINGS
# ----------------------------------------

print("\nAnalyzing event correlation...")

event_dates = timeline_df[
    [
        "date",
        "event_type_name"
    ]
].drop_duplicates()

plt.figure(figsize=(16, 7))

plt.plot(
    grossing_trend["date"],
    grossing_trend[
        "store_product_rank_grossing"
    ],
    label="Grossing Rank"
)

plt.gca().invert_yaxis()

for _, row in event_dates.iterrows():

    plt.axvline(
        row["date"],
        linestyle="--",
        alpha=0.3
    )

plt.title("Grossing Rank vs Timeline Events")

plt.xlabel("Date")
plt.ylabel("Grossing Rank")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR /
    "grossing_rank_vs_timeline_events.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Event correlation chart exported")

# ----------------------------------------
# EXECUTIVE INSIGHTS
# ----------------------------------------

print("\nGenerating executive summaries...")

print("\nGrossing Rank Summary:")
print(
    ranks_df[
        "store_product_rank_grossing"
    ].describe()
)

print("\nAverage Rating Summary:")
print(
    ratings_df[
        "average_star_cumulative"
    ].describe()
)

print("\nReview Sentiment Distribution:")
print(
    reviews_df[
        "review_sentiment"
    ].value_counts(normalize=True) * 100
)

print("\nTop Review Topics:")
print(
    reviews_df["topics"]
    .value_counts()
    .head(10)
)

# ----------------------------------------
# FINAL
# ----------------------------------------

print("\nEDA pipeline completed successfully")

print(
    f"\nExecution time: "
    f"{time.time() - start_time:.2f} seconds"
)

# Initial Business Insights

## Insight 1 — Store performance
MONOPOLY GO! maintained strong grossing rank performance across the period, with visible volatility around major app updates and seasonal content changes.

## Insight 2 — Product updates and ranking movement
Several new version releases and app store metadata changes happened close to ranking movements, suggesting that product updates and live-ops events may influence store performance.

## Insight 3 — User sentiment
Review sentiment is mostly driven by rating_value. Positive reviews indicate continued engagement, while negative reviews should be analyzed by topic to identify friction points.

## Insight 4 — Ratings stability
Cumulative average rating appears stable over time, which suggests the app has a strong rating base even when incremental reviews fluctuate.

## Insight 5 — Strategic opportunity
The client should monitor ranking movement and review sentiment within 7 days after major updates to quickly detect whether new releases are improving engagement or creating user friction.

In [ ]:
print("Grossing rank summary:")
print(ranks_df["store_product_rank_grossing"].describe())

print("\nAverage rating summary:")
print(ratings_df["average_star_cumulative"].describe())

print("\nReview sentiment distribution:")
print(reviews_df["review_sentiment"].value_counts(normalize=True) * 100)

print("\nTop review topics:")
print(reviews_df["topics"].value_counts().head(10))